In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = Path(r"C:\Users\Tushar\OneDrive\Desktop\tushar\Student folder")

def load_trip(trip_dir):
    tid = trip_dir.name
    track = pd.read_csv(trip_dir / f"{tid}.track", sep=r"\s+", header=None,
                        names=["lon", "lat", "t"])
    nodes = pd.read_csv(trip_dir / f"{tid}.nodes", sep=r"\s+", header=None,
                        names=["lon", "lat"])
    arcs = pd.read_csv(trip_dir / f"{tid}.arcs", sep=r"\s+", header=None,
                       names=["from_node", "to_node"])
    rp = trip_dir / f"{tid}.route"
    route = (pd.read_csv(rp, header=None, names=["arc_id"])["arc_id"].to_numpy()
             if rp.exists() else None)
    return {"id": tid, "track": track, "nodes": nodes, "arcs": arcs, "route": route}

t = load_trip(DATA_ROOT / "train" / "00000001")
print(len(t["track"]), len(t["nodes"]), len(t["arcs"]), len(t["route"]))
t["track"].head()

1315 88809 165790 525


,lon,lat,t
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0


In [2]:
rows = []
for part in ["train", "student_test"]:
    for d in sorted((DATA_ROOT / part).iterdir()):
        if not d.is_dir():
            continue
        t = load_trip(d)
        tr = t["track"]
        dt = np.diff(tr["t"].to_numpy())
        rows.append({
            "partition": part, "trip": t["id"],
            "n_gps": len(tr), "n_nodes": len(t["nodes"]), "n_arcs": len(t["arcs"]),
            "n_route": None if t["route"] is None else len(t["route"]),
            "gps_missing": int(tr.isna().sum().sum()),
            "t_not_increasing": int((dt <= 0).sum()),
            "pct_1s": round(float((dt == 1).mean()) * 100, 1),
            "max_gap_s": float(dt.max()),
            "lon_min": tr.lon.min(), "lon_max": tr.lon.max(),
            "lat_min": tr.lat.min(), "lat_max": tr.lat.max(),
            "route_ids_valid": None if t["route"] is None
                               else bool(t["route"].max() < len(t["arcs"])),
        })
inv = pd.DataFrame(rows)
print(inv.groupby("partition").size())
inv.to_csv("../experiments/dataset_inventory.csv", index=False)
inv.describe().T

partition
student_test    10
train           80
dtype: int64


,count,mean,std,min,25%,50%,75%,max
n_gps,90.0,2528.633333,3156.999879,262.000000,899.250000,1496.500000,2816.250000,19313.000000
n_nodes,90.0,22490.822222,30868.526339,299.000000,3655.250000,9305.000000,26296.500000,155003.000000
n_arcs,90.0,45132.722222,61920.284604,649.000000,7282.750000,18761.500000,52913.250000,307392.000000
n_route,90.0,386.811111,359.238455,59.000000,166.750000,272.500000,464.250000,2266.000000
gps_missing,90.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
t_not_increasing,90.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pct_1s,90.0,94.316667,11.227507,53.100000,96.400000,99.150000,100.000000,100.000000
max_gap_s,90.0,2408.522222,16996.219986,1.000000,1.000000,7.000000,67.250000,153064.000000
lon_min,90.0,17.110279,47.941128,-100.418538,-0.099509,16.163069,39.220636,137.197657
lon_max,90.0,17.293414,47.954191,-100.268049,0.214175,16.212317,39.256780,137.237029


In [3]:
print(inv.sort_values("max_gap_s", ascending=False)
        [["partition", "trip", "n_gps", "max_gap_s", "pct_1s"]].head(10))
print(inv.sort_values("pct_1s")
        [["partition", "trip", "n_gps", "pct_1s", "max_gap_s"]].head(10))
print(inv["route_ids_valid"].value_counts(dropna=False))

   partition      trip  n_gps  max_gap_s  pct_1s
72     train  00000090    536   153064.0    58.1
61     train  00000078   2909    52810.0    75.5
55     train  00000069   2067     2658.0    98.9
16     train  00000021  15783     2161.0   100.0
8      train  00000011   2828      909.0   100.0
34     train  00000042   7739      670.0    99.0
64     train  00000081   3224      536.0    72.0
18     train  00000024    801      458.0    97.1
36     train  00000044    546      341.0    90.1
67     train  00000084  16031      312.0    99.7
   partition      trip  n_gps  pct_1s  max_gap_s
17     train  00000022    474    53.1       22.0
3      train  00000004   1064    53.8       51.0
72     train  00000090    536    58.1   153064.0
32     train  00000039    363    58.3       10.0
38     train  00000046   1401    62.2       22.0
19     train  00000025   1179    69.4        2.0
64     train  00000081   3224    72.0      536.0
61     train  00000078   2909    75.5    52810.0
78     train  000000

In [4]:
import folium

t = load_trip(DATA_ROOT / "train" / "00000001")
nodes, arcs, track = t["nodes"], t["arcs"], t["track"]

# reference route: arc ids -> node ids -> coordinates
a = arcs.iloc[t["route"]]
route_coords = []
for f, to in zip(a.from_node, a.to_node):
    route_coords.append([(nodes.lat[f], nodes.lon[f]), (nodes.lat[to], nodes.lon[to])])

m = folium.Map(location=[track.lat.mean(), track.lon.mean()], zoom_start=15)

fg_gps = folium.FeatureGroup(name="GPS points (blue)")
for r in track.itertuples():
    folium.CircleMarker((r.lat, r.lon), radius=2, color="blue", fill=True,
                        tooltip=f"t={r.t:.0f}s").add_to(fg_gps)
fg_gps.add_to(m)

fg_route = folium.FeatureGroup(name="Reference route (green)")
for seg in route_coords:
    folium.PolyLine(seg, color="green", weight=4).add_to(fg_route)
fg_route.add_to(m)

folium.LayerControl().add_to(m)
m.save("../experiments/map_train_00000001.html")
m